# 06 · O dado que mente

Aula **"Próximo: o dado que mente"**: acurácia inflada, a duplicata que vaza pela porta de trás, o
split por grupo, e "o que cabe no domínio". O notebook 01 já mostrou que a base **não tem ID de
paciente** e que 6,8 % dos raio-X de teste têm um quase gêmeo no treino. Aqui, três perguntas:

1. **Vazamento:** as redes acertam mais nos exames de teste que têm um gêmeo no treino?
2. **Split por grupo:** quanto a AUC muda se a divisão nunca separa dois exames quase iguais?
3. **Atalho:** a rede precisa ver o coração? Treinar com **o coração tapado** e com **só o coração**.
4. **O 0,80 sem o paciente dos dois lados:** a receita completa do notebook 05, treinada de novo num
   corte em que nenhum paciente do teste aparece no treino.

Cada treino da CNN pequena tem variação própria (pesos iniciais, ordem dos lotes). Para não confundir
essa variação com o efeito procurado, os experimentos 2 e 3 repetem o treino com **3 sementes**.

**Orçamento fixo.** São 15 treinos. Cada um vai até **40 épocas** (paciência 8), menos que as 150 do
notebook 05, para caber no tempo. Por isso a AUC absoluta aqui fica um pouco abaixo da do 05; o que
interessa é a **diferença entre variantes treinadas com o mesmo orçamento**. A pergunta do 0,80 em si
fica para a seção 4, com as 150 épocas.

In [ ]:
import sys
from pathlib import Path

# Permite importar `src` mesmo fora do ambiente do uv (ex.: Colab com o repositório clonado).
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import config
from src.data.dados import (baixar_base, carregar_imagens, dividir, dividir_aleatorio,
                            dividir_por_grupo, listar_imagens)
from src.model import avaliacao, cnn
from src.utils.io import configurar_estilo, salvar_figura, salvar_tabela
from src.visao import auditoria, ict

configurar_estilo()
SECAO = "06-o-dado-que-mente"
SEMENTES = [config.SEMENTE, 7, 2026]

# True refaz os 15 treinos dos experimentos 2 e 3 (≈ 1h40 no CPU). False reaproveita as rodadas já
# gravadas em reports/tables/06-o-dado-que-mente/ (split-rodadas.csv, atalho-rodadas.csv).
RETREINAR = False
TABELAS = config.TABLES / SECAO

baixar_base()
catalogo = dividir(listar_imagens())
imagens = carregar_imagens(catalogo["caminho"])
y = catalogo["classe"].to_numpy()
conjunto = catalogo["conjunto"].to_numpy()
tr, va, te = (conjunto == "treino"), (conjunto == "validacao"), (conjunto == "teste")
grupos = pd.read_csv(config.DATA_PROCESSED / "grupos.csv", sep=";").set_index("id").loc[catalogo["id"], "grupo"].to_numpy()
predicoes = pd.read_csv(config.DATA_PROCESSED / "predicoes.csv", sep=";").set_index("id").loc[catalogo["id"]]
redes = [c for c in predicoes.columns if c not in ("conjunto", "classe")]
print("Escores disponíveis:", redes)

def treinar_cnn(X, conj, semente, epocas=40, paciencia=8):
    keras.utils.set_random_seed(semente)
    modelo = cnn.construir_cnn()
    t, v = conj == "treino", conj == "validacao"
    cnn.treinar(modelo, X[t], y[t], X[v], y[v], epocas=epocas, paciencia=paciencia)
    return cnn.prever(modelo, X)

## 1. Os exames de teste com gêmeo no treino

Para cada raio-X de teste, o mais parecido no treino + validação (correlação das imagens reduzidas,
notebook 01). Com correlação ≥ 0,92, o exame tem um **gêmeo**: em geral a mesma pessoa em outro dia.
Se as redes decoram pacientes, devem acertar mais nesses exames.

In [ ]:
assin = auditoria.assinatura(imagens)
idx_te, idx_ref = np.where(te)[0], np.where(~te)[0]
viz, corr = auditoria.vizinho_mais_proximo(assin[idx_te], assin[idx_ref])
gemeo = corr >= config.LIMIAR_QUASE_DUPLICATA
rotulo_gemeo = y[idx_ref[viz]]
print(f"Exames de teste com gêmeo: {gemeo.sum()} de {len(gemeo)} ({gemeo.mean():.1%})")
print(f"Copiar o rótulo do gêmeo acerta {np.mean(rotulo_gemeo[gemeo] == y[idx_te][gemeo]):.1%} das vezes")

linhas = []
for nome in redes:
    e = predicoes[nome].to_numpy()[idx_te]
    for grupo_nome, sel in [("com gêmeo no treino", gemeo), ("sem gêmeo", ~gemeo)]:
        linhas.append({"modelo": nome, "exames": grupo_nome, "n": int(sel.sum()),
                       **avaliacao.auc_bootstrap(y[idx_te][sel], e[sel], n=1000)})
gemeos = pd.DataFrame(linhas).set_index(["modelo", "exames"])
salvar_tabela(gemeos, SECAO, "com-e-sem-gemeo")
gemeos.round(3)

## 2. Split aleatório contra split por grupo

A base inteira é redividida 60/20/20 de dois jeitos, com as mesmas proporções:

* **aleatório**, estratificado pela classe (dois exames do mesmo paciente podem cair em lados opostos);
* **por grupo** (`StratifiedGroupKFold`), usando os grupos de quase duplicatas do notebook 01: cada
  grupo cai inteiro num lado só.

A mesma CNN pequena é treinada em cada divisão, com 3 sementes, e avaliada no teste **da sua própria
divisão**. Se o vazamento infla a AUC, a divisão aleatória deve sair melhor.

In [ ]:
X = cnn.tensores(imagens)
arquivo_rodadas = TABELAS / "split-rodadas.csv"
rodadas = []
for semente in ([] if not RETREINAR and arquivo_rodadas.exists() else SEMENTES):
    for nome, conj in [("aleatório", dividir_aleatorio(catalogo, semente).to_numpy()),
                       ("por grupo", dividir_por_grupo(catalogo, grupos, semente).to_numpy())]:
        p = treinar_cnn(X, conj, semente)
        t = conj == "teste"
        mesmo_grupo = pd.Series(conj).groupby(grupos).transform("nunique").to_numpy() > 1
        rodadas.append({"divisao": nome, "semente": semente,
                        "auc_teste": avaliacao.auc_bootstrap(y[t], p[t], n=10)["auc"],
                        "teste_em_grupo_partido": float(mesmo_grupo[t].mean())})
        print(rodadas[-1])
if rodadas:
    rodadas = pd.DataFrame(rodadas)
    salvar_tabela(rodadas, SECAO, "split-rodadas", indice=False)
else:
    rodadas = pd.read_csv(arquivo_rodadas, sep=";", decimal=",", encoding="utf-8-sig")
    print("Rodadas reaproveitadas de", arquivo_rodadas.name)
resumo_split = rodadas.groupby("divisao").agg(auc_media=("auc_teste", "mean"), auc_desvio=("auc_teste", "std"),
                                              teste_em_grupo_partido=("teste_em_grupo_partido", "mean"))
salvar_tabela(resumo_split, SECAO, "split-resumo")
resumo_split.round(3)

## 3. A rede precisa ver o coração?

Uma **caixa fixa** cobre a região do coração: a mediana, no treino, das bordas que o ICT do notebook 04
mediu (com folga). A caixa é a **mesma para todos os exames**: uma caixa do tamanho de cada coração
entregaria o rótulo pela própria forma do retângulo. Três versões da base, mesma divisão do projeto:

* **original**;
* **coração tapado**: a caixa pintada com a média da imagem;
* **só o coração**: tudo fora da caixa pintado.

Se a rede com o coração tapado ainda separa bem as classes, ela aprendeu **outra coisa**: pulmões,
tipo de filme, fios, posição do paciente.

In [ ]:
medidas = [ict.medir_ict(im, **config.PARAMS_ICT) for im in imagens[tr]]
CAIXA, _ = ict.caixas_de_referencia(medidas)
print("Caixa do coração (x0, y0, x1, y1):", CAIXA)

versoes = {
    "original": imagens,
    "coração tapado": auditoria.tapar(imagens, CAIXA),
    "só o coração": auditoria.tapar(imagens, CAIXA, isolar=True),
}
i_ex = int(np.where(catalogo["id"] == "train/true/1298")[0][0])
fig, eixos = plt.subplots(1, 3, figsize=(10, 3.6))
for ax, (nome, ims) in zip(eixos, versoes.items()):
    ax.imshow(ims[i_ex], cmap="gray", vmin=0, vmax=255)
    ax.set_title(nome)
    ax.axis("off")
salvar_figura(fig, SECAO, "versoes-tapadas")
plt.show()

In [ ]:
arquivo_atalho = TABELAS / "atalho-rodadas.csv"
if not RETREINAR and arquivo_atalho.exists():
    atalho = pd.read_csv(arquivo_atalho, sep=";", decimal=",", encoding="utf-8-sig")
    print("Rodadas reaproveitadas de", arquivo_atalho.name)
else:
    atalho = []
    for nome, ims in versoes.items():
        Xv = cnn.tensores(ims)
        for semente in SEMENTES:
            p = treinar_cnn(Xv, conjunto, semente)
            atalho.append({"versao": nome, "semente": semente, **avaliacao.auc_bootstrap(y[te], p[te], n=500)})
            print(nome, semente, round(atalho[-1]["auc"], 3))
    atalho = pd.DataFrame(atalho)
resumo_atalho = atalho.groupby("versao", sort=False).agg(auc_media=("auc", "mean"), auc_desvio=("auc", "std"),
                                                         auc_min=("auc", "min"), auc_max=("auc", "max"))
salvar_tabela(atalho, SECAO, "atalho-rodadas", indice=False)
salvar_tabela(resumo_atalho, SECAO, "atalho-resumo")

fig, ax = plt.subplots(figsize=(8, 3.8))
for k, (nome, grupo_v) in enumerate(atalho.groupby("versao", sort=False)):
    ax.scatter(grupo_v["auc"], [k] * len(grupo_v), color=config.PALETA[k], s=60, zorder=3)
    ax.errorbar(grupo_v["auc"].mean(), k, xerr=grupo_v["auc"].std(), fmt="|", color="#333", ms=18)
ax.axvline(0.5, color="#999", ls="--", lw=1)
ax.set(yticks=range(len(versoes)), yticklabels=list(versoes), xlabel="AUC no teste (3 sementes)",
       title="Quanto a rede depende de ver o coração")
salvar_figura(fig, SECAO, "atalho", atalho)
plt.show()
resumo_atalho.round(3)

## 4. O 0,80 sem o paciente dos dois lados

O experimento 2 compara divisões com um orçamento de 40 épocas, em que a CNN fica perto de 0,69. Ele
não diz se o **0,80** da receita completa (notebook 05: 150 épocas, paciência 15) continua de pé
quando o mesmo paciente não atravessa o corte. E tirar os gêmeos só do teste muda a avaliação, não o
treino: a rede já viu aqueles pacientes.

Aqui o **teste oficial** fica intocado e saem do treino e da validação **todos os exames do mesmo grupo
de algum exame do teste** (grupos do notebook 01). A CNN pequena é treinada de novo com a receita do
notebook 05, nos dois cortes, com 3 sementes cada (no corte oficial, a semente 42 é o próprio modelo do
05). Os treinos rodam em `scripts/treinar_corte_limpo.py`, ≈ 27 min cada, e as previsões ficam em
`data/processed/corte_limpo.csv`.

Como o teste é o mesmo, a comparação é **pareada**: mesma semente, mesmos exames, só muda o treino. A
DenseNet do CheXpert congelada (notebook 11, AUC 0,826) também é refeita no corte limpo.

In [ ]:
from src.data.dados import sem_paciente_do_teste
from src.model import transferencia

corte_limpo = sem_paciente_do_teste(catalogo["conjunto"], grupos).to_numpy()
fora = corte_limpo == "fora"
print(f"Saem {fora.sum()} exames ({(fora & tr).sum()} do treino, {(fora & va).sum()} da validação; "
      f"{(y[fora] == 0).sum()} normais e {(y[fora] == 1).sum()} com cardiomegalia)")
mesmo_grupo_fora = np.isin(grupos[te], grupos[fora])
print(f"Exames do teste que tinham alguém do mesmo grupo no treino ou na validação: {mesmo_grupo_fora.sum()}")

ids_te = catalogo["id"].to_numpy()[te]
rodadas_cl = pd.read_csv(config.DATA_PROCESSED / "corte_limpo.csv", sep=";")
escores = {("oficial", config.SEMENTE): predicoes["CNN pequena · cru"].to_numpy()[te]}
for (corte, semente), g in rodadas_cl.groupby(["corte", "semente"]):
    escores[(corte, semente)] = g.set_index("id").loc[ids_te, "escore"].to_numpy()

sem_gemeo = ~gemeo  # os 76 exames de teste com gêmeo, da seção 1, saem nesta leitura
linhas = []
for (corte, semente), e in sorted(escores.items()):
    linhas.append({"corte": corte, "semente": semente,
                   **avaliacao.auc_bootstrap(y[te], e, n=1000),
                   "auc_sem_gemeos": avaliacao.auc_bootstrap(y[te][sem_gemeo], e[sem_gemeo], n=10)["auc"]})
cnn_cortes = pd.DataFrame(linhas)
salvar_tabela(cnn_cortes, SECAO, "corte-limpo-rodadas", indice=False)
cnn_cortes.round(3)

In [ ]:
# Pareado por semente: mesma inicialização, mesmo teste, só o treino muda.
pares = []
for semente in sorted({s for _, s in escores}):
    if ("oficial", semente) in escores and ("sem paciente do teste", semente) in escores:
        d = avaliacao.comparar_auc(y[te], escores[("sem paciente do teste", semente)], escores[("oficial", semente)])
        pares.append({"semente": semente, **d})
pares = pd.DataFrame(pares).set_index("semente")
resumo_cl = cnn_cortes.groupby("corte").agg(auc_media=("auc", "mean"), auc_desvio=("auc", "std"),
                                            auc_sem_gemeos=("auc_sem_gemeos", "mean"))

# DenseNet do CheXpert congelada (notebook 11): a mesma cabeça, treinada em cada corte.
feat = transferencia.extrair_com_cache("DenseNet121 · CheXpert", catalogo["id"].to_numpy(), imagens)
densenet = {}
for corte, conj in [("oficial", conjunto), ("sem paciente do teste", corte_limpo)]:
    t = conj == "treino"
    densenet[corte] = transferencia.cabeca(5).fit(feat[t], y[t]).predict_proba(feat[te])[:, 1]
d_dn = avaliacao.comparar_auc(y[te], densenet["sem paciente do teste"], densenet["oficial"])
dn = pd.DataFrame({c: {**avaliacao.auc_bootstrap(y[te], e, n=1000),
                       "auc_sem_gemeos": avaliacao.auc_bootstrap(y[te][sem_gemeo], e[sem_gemeo], n=10)["auc"]}
                   for c, e in densenet.items()}).T

salvar_tabela(pares, SECAO, "corte-limpo-pareado")
salvar_tabela(resumo_cl, SECAO, "corte-limpo-resumo")
salvar_tabela(dn, SECAO, "corte-limpo-densenet")
print("CNN pequena, 150 épocas, por corte (3 sementes):")
print(resumo_cl.round(3))
print("\nDiferença sem paciente do teste − oficial, pareada por semente:")
print(pares.round(3))
print(f"\nDenseNet do CheXpert: oficial {dn.loc['oficial', 'auc']:.3f} · sem paciente do teste "
      f"{dn.loc['sem paciente do teste', 'auc']:.3f} · diferença {d_dn['diferenca']:+.3f} "
      f"(IC {d_dn['ic_baixo']:+.3f} a {d_dn['ic_alto']:+.3f})")

fig, ax = plt.subplots(figsize=(8, 3.4))
for k, (corte, g) in enumerate(cnn_cortes.groupby("corte", sort=False)):
    ax.scatter(g["auc"], [k] * len(g), color=config.PALETA[k], s=60, zorder=3, label=f"CNN · {corte}")
    for _, r in g.iterrows():
        ax.annotate(str(r["semente"]), (r["auc"], k), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=7)
    ax.scatter(dn.loc[corte, "auc"], k, marker="D", color="#333", s=40, zorder=4)
ax.scatter([], [], marker="D", color="#333", label="DenseNet do CheXpert")
ax.set(yticks=[0, 1], yticklabels=list(cnn_cortes.groupby("corte", sort=False).groups), ylim=(-0.6, 1.6),
       xlabel="AUC no teste oficial (1.114 exames)", title="O mesmo teste, com e sem o paciente dos dois lados")
ax.legend(fontsize=8, loc="lower left")
salvar_figura(fig, SECAO, "corte-limpo", cnn_cortes)
plt.show()

## Conclusões

* **Os exames com gêmeo são mais fáceis, não decorados:** nos 76 exames de teste com um quase gêmeo
  no treino, a AUC sobe em todos os modelos: CNN crua 0,80 → 0,86, MobileNet 0,79 → 0,90, e também o
  **ICT clássico, 0,73 → 0,86**. O ICT não aprende nada dos dados, então a subida não vem de
  memorizar pacientes: esse subconjunto é simplesmente mais fácil (exames repetidos tendem a ser de
  doentes acompanhados, com achados mais claros). Com n = 76, os intervalos são largos.
* **Com 40 épocas, agrupar não muda a AUC:** a divisão aleatória dá 0,680 ± 0,033 e a por grupo
  0,694 ± 0,013. Mas nesse orçamento a CNN fica perto de 0,69, então o experimento não diz nada sobre
  o 0,80 da receita completa. Para isso serve a seção 4.
* **O 0,80 continua de pé sem o paciente dos dois lados:** tirando do treino e da validação os 169
  exames do mesmo grupo de algum exame do teste, a CNN de 150 épocas dá **0,807 ± 0,008** em 3
  sementes, contra **0,806 ± 0,014** no corte oficial, no mesmo teste oficial. Pareadas por semente, as
  diferenças vão de −0,011 a +0,011, todas com intervalo cruzando zero e menores que a variação entre
  sementes do próprio corte oficial (0,793 a 0,821). Sem os 76 gêmeos no teste, 0,801 contra 0,799. A
  DenseNet do CheXpert dá 0,826 nos dois cortes. O vazamento por paciente existe (82 exames do teste
  tinham alguém do mesmo grupo no treino), mas não sustenta a AUC.
* **A rede não depende só do coração:** com a região do coração tapada, a AUC cai de 0,695 para
  **0,657**, só 0,04. Cerca de 80 % do que a rede separa acima do acaso (0,157 de 0,195) sobrevive sem ver a caixa do
  coração. A informação também está nos pulmões, no mediastino acima da caixa, no diafragma e no que
  não é anatomia (tipo de filme, fios, posição).
* **Só o coração é instável:** com tudo fora da caixa tapado, as três sementes deram 0,75, 0,60 e
  0,52. A região do coração sozinha tem sinal (a melhor rodada passa a original), mas com 40 épocas
  o treino nem sempre o encontra.
* **O rótulo tem ruído:** copiar o rótulo do gêmeo acerta só 69,7 %. Pares que são, na maioria, a
  mesma pessoa discordam em 30 % dos casos.

**O dado mente pouco aqui, mas mente:** o vazamento por paciente não inflou a AUC, nem no orçamento
curto nem na receita completa, e o ruído de
rótulo põe um teto em qualquer modelo. A lição mais forte é a do coração tapado: uma AUC alta não
garante que a rede usa o critério do médico.